# HiddenBench concrete FullPrompt and provider boundary

This migrated Phase 3/4 notebook uses benchmark-owned concrete full prompts. It does not construct a universal prompt context and it does not imply a complete HiddenBench game runner.

In [ ]:
from pathlib import Path
from mas_cc.cli.prompt import _hiddenbench_values
from mas_cc.llm_providers import CompletionRequest
from mas_cc.prompts import RegexTokenCounter
from mas_cc.prompts.plugins.hidden_profile_v3 import hidden_profile_discussion_prompt, hidden_profile_vote_prompt
data_path = Path('scripts/local_llms/hiddenbench_population_pipeline/data/hiddenbench/scaled/exact_replication/N_32.json')
if not data_path.exists(): data_path = Path('../scripts/local_llms/hiddenbench_population_pipeline/data/hiddenbench/scaled/exact_replication/N_32.json')
data_path = data_path.resolve()
values = _hiddenbench_values(data_path, task_id=1, agent_id=0)


In [ ]:
discussion = hidden_profile_discussion_prompt().bind(scenario=values['scenario'], private_information={'information': values['information']}, transcript=values['transcript'])
vote_definition = hidden_profile_vote_prompt()
vote = type(vote_definition)(vote_definition.family, vote_definition.version, vote_definition.blocks, type(vote_definition.response_contract)('json_vote', values['answers']), vote_definition.message_mode, vote_definition.block_separator).bind(scenario=values['scenario'], private_information={'information': values['information']}, transcript=values['transcript'])
discussion_compiled = discussion.compile(RegexTokenCounter())
vote_compiled = vote.compile(RegexTokenCounter())
assert discussion.definition_hash != vote.definition_hash
assert values['metadata']['audit_answer_included'] is False
[block.to_dict() for block in discussion_compiled.blocks]


## Normalized requests
Only compiled messages cross this boundary. Prompt values and local fingerprints remain request metadata and are excluded by `wire_messages()`.

In [ ]:
discussion_request = CompletionRequest(discussion_compiled.messages, max_output_tokens=128, metadata={'prompt_family': discussion_compiled.family, 'definition_hash': discussion_compiled.definition_hash, 'instance_hash': discussion_compiled.instance_hash})
vote_request = CompletionRequest(vote_compiled.messages, max_output_tokens=128, metadata={'prompt_family': vote_compiled.family, 'definition_hash': vote_compiled.definition_hash, 'instance_hash': vote_compiled.instance_hash})
assert all(set(message) == {'role', 'content'} for message in discussion_request.wire_messages())
{'discussion': discussion_request.wire_messages(), 'vote': vote_request.wire_messages()}
